# Détection de maladies foliaires par YOLO — Notebook de finetuning

Fine-tuning d'un modèle YOLO (Ultralytics) sur le dataset [FieldPlant](https://universe.roboflow.com/plant-disease-detection/fieldplant) pour la détection et classification de maladies foliaires en conditions de terrain.

**Plan**
1. Chargement du dataset & EDA
2. Entraînement (comparaison YOLO nano vs small)
3. Évaluation quantitative (mAP, P/R/F1, matrice de confusion, courbes PR)
4. Compromis précision/vitesse (temps d'inférence, taille du modèle)
5. Exemples qualitatifs (bonnes détections, faux positifs, faux négatifs)
6. Conclusion

In [ ]:
import os
import time
import glob
import yaml
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from ultralytics import YOLO
import torch

print('Torch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DATA_DIR = Path('../data')
RUNS_DIR = Path('../runs')
FIG_DIR = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement du dataset & EDA

In [ ]:
# Le zip Roboflow (format YOLOv8) doit être dézippé dans data/
# Structure attendue : data/{train,valid,test}/{images,labels} + data/data.yaml
#
# Note : l'export Roboflow v11 ne contenait qu'un split "train" (5156 images), sans valid/test
# malgré les clés présentes dans data.yaml. Le split train/valid/test (80/10/10) a été recréé
# manuellement via src/split_dataset.py (seed=42) avant d'exécuter ce notebook.
DATA_YAML = DATA_DIR / 'data.yaml'

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print('Classes :', data_cfg['names'])
print('Nombre de classes :', data_cfg['nc'])
data_cfg

In [ ]:
# Nombre d'images par split
for split in ['train', 'valid', 'test']:
    img_dir = DATA_DIR / split / 'images'
    if img_dir.exists():
        n = len(list(img_dir.glob('*')))
        print(f'{split}: {n} images')

In [ ]:
# Distribution des classes (basée sur les labels d'entraînement)
class_names = data_cfg['names']
counts = {c: 0 for c in class_names}

label_dir = DATA_DIR / 'train' / 'labels'
for label_file in label_dir.glob('*.txt'):
    with open(label_file, 'r') as f:
        for line in f:
            cls_id = int(line.split()[0])
            counts[class_names[cls_id]] += 1

df_counts = pd.DataFrame(list(counts.items()), columns=['classe', 'nb_annotations']).sort_values('nb_annotations', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_counts, x='nb_annotations', y='classe')
plt.title('Distribution des annotations par classe (train)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distribution.png', dpi=150)
plt.show()
df_counts

In [ ]:
# Aperçu de quelques images du dataset
sample_imgs = random.sample(list((DATA_DIR / 'train' / 'images').glob('*')), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    ax.imshow(Image.open(img_path))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Entraînement — comparaison YOLO nano vs small

On fine-tune deux tailles de modèle pour discuter du compromis précision/vitesse.

In [ ]:
EPOCHS = 100
IMG_SIZE = 640
BATCH = 16

# Note performance (Windows) : le chargement par défaut (workers multiprocess + lecture JPEG
# à chaque itération) s'est révélé ~6x plus lent que nécessaire, probablement à cause du scan
# temps réel de l'antivirus sur des milliers de petits fichiers. cache='ram' + workers=0 (pour
# éviter un bug connu de Windows sur le pickling d'objets >2 Go lors du spawn de sous-processus)
# règle le problème.
model_n = YOLO('yolov8n.pt')
results_n = model_n.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(RUNS_DIR / 'detect'),
    name='train_nano',
    patience=20,
    cache='ram',
    workers=0,
)

In [ ]:
model_s = YOLO('yolov8s.pt')
results_s = model_s.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(RUNS_DIR / 'detect'),
    name='train_small',
    patience=20,
    cache='ram',
    workers=0,
)

## 3. Évaluation quantitative

In [ ]:
# Charger les meilleurs poids et évaluer sur le split de validation/test
best_n = YOLO(str(RUNS_DIR / 'detect' / 'train_nano' / 'weights' / 'best.pt'))
best_s = YOLO(str(RUNS_DIR / 'detect' / 'train_small' / 'weights' / 'best.pt'))

metrics_n = best_n.val(data=str(DATA_YAML), split='test')
metrics_s = best_s.val(data=str(DATA_YAML), split='test')

In [ ]:
# mAP@0.5 et mAP@0.5:0.95
comparison = pd.DataFrame({
    'modèle': ['YOLOv8n', 'YOLOv8s'],
    'mAP@0.5': [metrics_n.box.map50, metrics_s.box.map50],
    'mAP@0.5:0.95': [metrics_n.box.map, metrics_s.box.map],
})
comparison

In [ ]:
# Précision / Rappel / F1-score par classe
def per_class_report(metrics, class_names, model_name):
    p, r, f1 = metrics.box.p, metrics.box.r, metrics.box.f1
    df = pd.DataFrame({
        'classe': class_names,
        'précision': p,
        'rappel': r,
        'f1': f1,
    })
    df['modèle'] = model_name
    return df

report_n = per_class_report(metrics_n, class_names, 'YOLOv8n')
report_s = per_class_report(metrics_s, class_names, 'YOLOv8s')
full_report = pd.concat([report_n, report_s]).sort_values(['classe', 'modèle'])
full_report

In [ ]:
# Matrice de confusion (générée automatiquement par Ultralytics dans le dossier de run)
from IPython.display import Image as IPImage, display

display(IPImage(filename=str(RUNS_DIR / 'detect' / 'train_small' / 'confusion_matrix.png')))

In [ ]:
# Courbes Précision-Rappel par classe
display(IPImage(filename=str(RUNS_DIR / 'detect' / 'train_small' / 'PR_curve.png')))

## 4. Compromis précision / vitesse

In [ ]:
import os

def benchmark_inference(model, n_runs=50, imgsz=640):
    dummy = torch.rand(1, 3, imgsz, imgsz).to(model.device)
    # warmup
    for _ in range(5):
        model.predict(dummy, verbose=False)
    start = time.time()
    for _ in range(n_runs):
        model.predict(dummy, verbose=False)
    elapsed = (time.time() - start) / n_runs * 1000
    return elapsed

def model_size_mb(weights_path):
    return os.path.getsize(weights_path) / (1024 * 1024)

speed_n = benchmark_inference(best_n)
speed_s = benchmark_inference(best_s)

size_n = model_size_mb(RUNS_DIR / 'detect' / 'train_nano' / 'weights' / 'best.pt')
size_s = model_size_mb(RUNS_DIR / 'detect' / 'train_small' / 'weights' / 'best.pt')

speed_comparison = pd.DataFrame({
    'modèle': ['YOLOv8n', 'YOLOv8s'],
    'temps_inférence_ms': [speed_n, speed_s],
    'taille_mo': [size_n, size_s],
})
speed_comparison

## 5. Exemples qualitatifs

Bonnes détections, faux positifs, faux négatifs — à commenter (ex. confusion entre deux maladies similaires, échec sur symptômes minuscules).

In [ ]:
test_images = list((DATA_DIR / 'test' / 'images').glob('*'))[:8]

results = best_s.predict(source=test_images, conf=0.25, save=True, project=str(RUNS_DIR / 'detect'), name='predictions_qualitatives')

# Parcourir les résultats et afficher les détections
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, r in zip(axes.flatten(), results):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.axis('off')
plt.tight_layout()
plt.show()

**Observations sur les exemples ci-dessus** (voir `reports/rapport.md` section 7 pour l'analyse détaillée avec comparaison vérité terrain vs prédiction) :

- **Bonne détection** : lésions de *Tomato blight leaf* et feuille *Tomato healthy* bien localisées, boîtes alignées avec la vérité terrain.
- **Faux positifs** : le modèle a tendance à ajouter des boîtes *Tomato healthy* supplémentaires non annotées autour d'une zone infectée (sur-segmentation).
- **Faux négatif** : lésion de *Corn leaf blight* en bordure d'image (bas du cadre) non détectée — échec typique sur les symptômes petits/périphériques.
- **Confusion inter-classes** : concentrée entre maladies du maïs aux symptômes visuellement proches (lésions allongées brun-jaune), quasi inexistante entre cultures différentes (manioc/maïs/tomate).

## 6. Conclusion

- YOLOv8n (70 époques, early stop) et YOLOv8s (53 époques, early stop) atteignent des performances quasi identiques : mAP@0.5 ≈ 0,73 (n) / 0,74 (s), mAP@0.5:0.95 ≈ 0,58 (n) / 0,57 (s).
- YOLOv8s n'apporte donc aucun gain net malgré 3,7x plus de paramètres et un fichier 3,6x plus lourd (21,5 Mo vs 5,9 Mo) — **YOLOv8n est recommandé** comme modèle final, notamment pour un déploiement terrain/mobile.
- La limite principale n'est pas le modèle mais le **dataset** : fort déséquilibre de classes (3 classes couvrent l'essentiel des annotations, plusieurs classes ont moins de 20 exemples, et *Corn Charcoal* n'a qu'1 seule annotation dans tout le dataset — impossible à apprendre et évaluer correctement).
- Pistes d'amélioration : augmentation de données ciblée sur les classes rares, split stratifié par classe, fusion des classes de maïs les plus confondues entre elles.

Voir `reports/rapport.md` pour le rapport d'expérimentation complet.